# 29 - Longer history: today's model on 12-13 onwards (P9-03, D9-1)

The owner approved D9-1 on 2026-11-07: Phase 9 research trains on the longer history, and the tune seasons widen.

- **Data:** `data/processed/all_matches_history.csv` (`python src/load_data.py --no-refresh --history-start 10-11`).
  Features are built from 10-11. Training rows start at **12-13** (10-11/11-12 only warm up ratings and form).
  **19-20 and 20-21 are never training or test rows** (D3), but they still carry ratings and form forward.
- **xG before 20-21 is estimated** (P9-03 Outcome): a FotMob-stats estimate (box shots, on target, big chances;
  R^2 0.86) for 16-17..19-20, a shots estimate (R^2 0.56) for 10-11..15-16, and **none** for P1 before 17-18 and
  N1 before 16-17 (there `XGDDiffLast5` is 0).
- **Model:** today's production setup, unchanged (pooled LR + league indicators, E0 its own model,
  `CalibratedClassifierCV` sigmoid). Only the training history changes.

Two questions:
1. **Like-for-like:** on the existing tune seasons 22-23/23-24, is today's model better when trained on 12-13 onwards
   than on 21-22 onwards? (paired bootstrap, 5,000, seed 42; *better* = CI upper < 0)
2. **Widened tune set** 15-16..18-19 + 21-22..23-24: the baseline (and market) every later P9 ticket compares against.

The report seasons (24-25, 25-26) and 26-27 stay sealed.

In [1]:
import os
import sys
import time
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

import model_lab as lab
from evaluate import per_match_log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

TUNE = lab.PERIODS["tune"]
WIDE = lab.PERIODS["tune_wide"]
TICKET = "P9-03"
WIDE, TUNE

(['15-16', '16-17', '17-18', '18-19', '21-22', '22-23', '23-24'],
 ['22-23', '23-24'])

## Data

In [2]:
t0 = time.time()
hist = lab.history_matches()
hframe = lab.history_frame()
print(f"history_matches: {len(hist)} rows in {time.time() - t0:.0f} s")

summary = (
    hist.assign(XGSource=hist["XGSource"].fillna("none"))
    .pivot_table(index="Season", columns="XGSource", values="FTR", aggfunc="size", fill_value=0)
)
summary.insert(0, "matches", summary.sum(axis=1))
summary["market_from"] = hframe.groupby("Season")["qSource"].agg(
    lambda s: ", ".join(f"{k} {v}" for k, v in s.value_counts().items())
)
summary

history_matches: 29543 rows in 19 s


XGSource,matches,FotMob,Understat,none,shots-proxy,stats-proxy,market_from
Season,,,,,,,
12-13,2372,0,0,547,1825,0,"pinnacle 2338, avg 34"
13-14,2372,0,0,546,1826,0,"pinnacle 2365, avg 7"
14-15,2438,0,0,612,1826,0,"pinnacle 2429, avg 9"
15-16,2438,0,0,612,1826,0,"pinnacle 2435, avg 3"
16-17,2438,0,0,306,306,1826,"pinnacle 2429, avg 9"
17-18,2438,0,0,0,1,2437,"pinnacle 2412, avg 26"
18-19,2438,0,0,0,1,2437,"pinnacle 2424, avg 14"
21-22,2438,2438,0,0,0,0,"pinnacle 2436, avg 1"
22-23,2438,2438,0,0,0,0,pinnacle 2438


Sanity check: on 22-23/23-24 the history frame should have every match in today's lab frame, with the same
results and the same market probabilities (the odds columns are the same; only older seasons were added).

In [3]:
today = lab.lab_frame()
keys = today.index[today["Season"].isin(TUNE)]
missing = keys.difference(hframe.index)
print("tune matches in today's frame:", len(keys), "| missing from history frame:", len(missing))
print("results identical:", (hframe.loc[keys, "FTR"] == today.loc[keys, "FTR"]).all())
print("max |q diff|:", float(np.abs(hframe.loc[keys, lab.MARKET].to_numpy() - today.loc[keys, lab.MARKET].to_numpy()).max()))

tune matches in today's frame: 4802 | missing from history frame: 0
results identical: True
max |q diff|: 0.0


## 1. Like-for-like on 22-23/23-24

`reference` = today's production model trained from 21-22 (the lab's baseline, which reproduces
`oos_predictions.csv`). `candidate` = the same model with the longer history as training data and warm-up.

In [4]:
t0 = time.time()
reference = lab.candidate_oos(lab.production, TUNE)
print("reference vs oos_predictions.csv: max |diff| =", f"{lab.check_production(reference):.1e}")

candidate = lab.candidate_oos(lab.production, TUNE, matches=hist)
print(f"walk-forwards in {time.time() - t0:.0f} s; candidate trained through:",
      candidate.groupby("Season")["TrainedThrough"].first().to_dict())

result = lab.compare(candidate, reference)
result

reference vs oos_predictions.csv: max |diff| = 0.0e+00


walk-forwards in 17 s; candidate trained through: {'22-23': '21-22', '23-24': '22-23'}


{'period': 'tune',
 'n': 4802,
 'candidate': 0.9660240688681362,
 'reference': 0.9696781250249311,
 'diff': -0.0036540561567948854,
 'ci': (-0.006504570046419632, -0.0008227450349442678),
 'seasons_better': 2,
 'n_seasons': 2}

In [5]:
cand_scores = lab.score(candidate)
ref_scores = lab.score(reference)
cols = ["n", "log_loss", "market_log_loss", "gap", "gap_ci_low", "gap_ci_high", "accuracy",
        "slope_D", "mean_pD", "actual_D", "model_share", "residual_weight", "residual_gain"]
pd.concat({"today (from 21-22)": ref_scores[cols], "history (from 12-13)": cand_scores[cols]}, axis=1).T.round(4)

league                                      all        D1        E0        F1        I1        N1        P1       SP1
today (from 21-22)   n                4802.0000  612.0000  760.0000  686.0000  760.0000  612.0000  612.0000  760.0000
                     log_loss            0.9697    0.9880    0.9538    1.0041    0.9952    0.9268    0.9286    0.9819
                     market_log_loss     0.9503    0.9707    0.9374    0.9898    0.9733    0.9159    0.8900    0.9643
                     gap                 0.0194    0.0173    0.0164    0.0143    0.0218    0.0110    0.0386    0.0176
                     gap_ci_low          0.0142    0.0011    0.0015    0.0030    0.0086   -0.0060    0.0251    0.0047
                     gap_ci_high         0.0247    0.0330    0.0315    0.0262    0.0352    0.0270    0.0524    0.0298
                     accuracy            0.5452    0.5163    0.5697    0.5190    0.5263    0.5784    0.5735    0.5368
                     slope_D             1.9451    3.5880    1.6096    2.9765    1.2783    3.2674    2.9205    1.8087
                     mean_pD             0.2466    0.2440    0.2306    0.2526    0.2510    0.2381    0.2504    0.2584
                     actual_D            0.2468    0.2549    0.2224    0.2522    0.2789    0.2435    0.2124    0.2579
                     model_share         0.0782    0.0782    0.0782    0.0782    0.0782    0.0782    0.0782    0.0782
                     residual_weight     0.1212    0.1212    0.1212    0.1212    0.1212    0.1212    0.1212    0.1212
                     residual_gain      -0.0002   -0.0003   -0.0009    0.0002    0.0002   -0.0021    0.0015   -0.0003
history (from 12-13) n                4802.0000  612.0000  760.0000  686.0000  760.0000  612.0000  612.0000  760.0000
                     log_loss            0.9660    0.9836    0.9524    1.0072    0.9880    0.9185    0.9216    0.9804
                     market_log_loss     0.9503    0.9707    0.9374    0.9898    0.9733    0.9159    0.8900    0.9643
                     gap                 0.0157    0.0129    0.0151    0.0174    0.0146    0.0027    0.0316    0.0161
                     gap_ci_low          0.0109   -0.0016    0.0018    0.0040    0.0026   -0.0126    0.0190    0.0036
                     gap_ci_high         0.0208    0.0279    0.0284    0.0308    0.0264    0.0176    0.0445    0.0285
                     accuracy            0.5473    0.5163    0.5645    0.5160    0.5342    0.5948    0.5752    0.5355
                     slope_D             3.2712    3.0036    2.0752    2.9327    2.6855    3.9407    6.9864    4.2690
                     mean_pD             0.2457    0.2434    0.2379    0.2567    0.2507    0.2402    0.2450    0.2453
                     actual_D            0.2468    0.2549    0.2224    0.2522    0.2789    0.2435    0.2124    0.2579
                     model_share         0.0629    0.0629    0.0629    0.0629    0.0629    0.0629    0.0629    0.0629
                     residual_weight     0.0861    0.0861    0.0861    0.0861    0.0861    0.0861    0.0861    0.0861
                     residual_gain      -0.0001   -0.0003   -0.0001    0.0001    0.0002   -0.0016    0.0010   -0.0001

In [6]:
# Per season and per league: candidate - reference log loss (negative = history better)
frame = lab.lab_frame()
rows = frame[frame["Season"].isin(TUNE)]
y = rows["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
diff = (per_match_log_loss(y, candidate.loc[rows.index, lab.PROB].to_numpy())
        - per_match_log_loss(y, reference.loc[rows.index, lab.PROB].to_numpy()))
pd.DataFrame({"Season": rows["Season"], "Div": rows["Div"], "diff": diff}).pivot_table(
    index="Div", columns="Season", values="diff", aggfunc="mean", margins=True
).round(4)

Season,22-23,23-24,All
Div,,,
D1,-0.0060,-0.0028,-0.0044
E0,-0.0019,-0.0007,-0.0013
F1,0.0006,0.0061,0.0031
I1,-0.0073,-0.0071,-0.0072
N1,-0.0155,-0.0011,-0.0083
P1,-0.0095,-0.0044,-0.0070
SP1,-0.0079,0.0049,-0.0015
All,-0.0065,-0.0008,-0.0037


In [7]:
lab.log_variant(cand_scores, TICKET, "production-history",
                description="today's model, unchanged, trained on 12-13 onwards (10-11/11-12 warm-up, COVID out)",
                params={"history_start": "10-11", "train_start": lab.HISTORY_TRAIN_START})
lab.load_variants().query("ticket == @TICKET")

,ticket,name,description,params,period,n,log_loss,accuracy,brier,market_log_loss,gap,gap_ci_low,gap_ci_high,gap_avg,draw_slope,model_share,blend_gap,residual_weight,residual_weight_ci_low,residual_weight_ci_high,residual_gain,residual_gain_ci_low,residual_gain_ci_high,timestamp
3,P9-03,production-history,"today's model, unchanged, trained on 12-13 onw...","{""history_start"": ""10-11"", ""train_start"": ""12-...",tune,4802,0.966024,0.547272,0.57419,0.950275,0.015749,0.010876,0.020838,0.015009,3.27119,0.062879,-0.001295,0.086103,-0.077034,0.249241,-0.000111,-0.000517,0.00031,2026-11-07T08:41:12+00:00


## 2. The widened tune set

Each season in 15-16..18-19 + 21-22..23-24 is predicted by a model trained on every eligible season before it
(from 12-13, without 19-20/20-21). The blend/residual information tests are still fitted on 22-23/23-24, as in the lab.

In [8]:
t0 = time.time()
wide = lab.candidate_oos(lab.production, WIDE, matches=hist)
elapsed = time.time() - t0
print(f"walk-forward over the widened tune set: {elapsed:.0f} s (acceptance: < 300 s)")

climate = lab.candidate_oos(lab.climatology, WIDE, matches=hist)
market = lab.market_proba(hframe)

wide_scores = {
    "production-history": lab.score(wide, frame=hframe, period="tune_wide"),
    "market": lab.score(market, frame=hframe, period="tune_wide"),
    "climatology": lab.score(climate, frame=hframe, period="tune_wide"),
}
pd.concat({k: v.loc[["all"], cols] for k, v in wide_scores.items()}).droplevel(1).round(4)

walk-forward over the widened tune set: 20 s (acceptance: < 300 s)


,n,log_loss,market_log_loss,gap,gap_ci_low,gap_ci_high,accuracy,slope_D,mean_pD,actual_D,model_share,residual_weight,residual_gain
production-history,16991,0.9723,0.9511,0.0213,0.0185,0.0240,0.5382,2.7503,0.2481,0.246,0.0629,0.0861,0.0003
market,16991,0.9511,0.9511,0.0000,0.0000,0.0000,0.5489,1.1366,0.2457,0.246,0.0000,0.0000,0.0000
climatology,16991,1.0664,0.9511,0.1153,0.1090,0.1218,0.4506,0.2195,0.2494,0.246,-0.1325,-0.2776,-0.0001


In [9]:
wide_scores["production-history"][cols].round(4)

,n,log_loss,market_log_loss,gap,gap_ci_low,gap_ci_high,accuracy,slope_D,mean_pD,actual_D,model_share,residual_weight,residual_gain
league,,,,,,,,,,,,,
all,16991,0.9723,0.9511,0.0213,0.0185,0.0240,0.5382,2.7503,0.2481,0.2460,0.0629,0.0861,0.0003
D1,2142,0.9942,0.9804,0.0139,0.0062,0.0218,0.5149,3.2550,0.2454,0.2474,0.0629,0.0861,-0.0001
E0,2660,0.9584,0.9412,0.0173,0.0102,0.0241,0.5571,1.4511,0.2442,0.2323,0.0629,0.0861,-0.0001
F1,2586,1.0091,0.9881,0.0210,0.0140,0.0278,0.5101,3.4526,0.2574,0.2641,0.0629,0.0861,0.0003
I1,2659,0.9712,0.9453,0.0258,0.0190,0.0326,0.5461,2.6615,0.2526,0.2542,0.0629,0.0861,0.0007
N1,2142,0.9463,0.9313,0.0150,0.0074,0.0228,0.5593,3.2672,0.2445,0.2302,0.0629,0.0861,-0.0002
P1,2142,0.9384,0.9062,0.0322,0.0249,0.0393,0.5574,5.0848,0.2496,0.2316,0.0629,0.0861,0.0010
SP1,2660,0.9823,0.9592,0.0231,0.0159,0.0304,0.5252,3.8441,0.2426,0.2571,0.0629,0.0861,0.0002


In [10]:
# Per season: model vs market (Pinnacle opening, Avg fallback) on the widened tune set
rows = hframe[hframe["Season"].isin(WIDE)]
y = rows["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
ll_model = per_match_log_loss(y, wide.loc[rows.index, lab.PROB].to_numpy())
ll_market = per_match_log_loss(y, rows[lab.MARKET].to_numpy())
by_season = pd.DataFrame({"Season": rows["Season"], "model": ll_model, "market": ll_market}).groupby("Season").mean()
by_season["gap"] = by_season["model"] - by_season["market"]
by_season["trained_seasons"] = [
    len([s for s in hist["Season"].unique() if s < season]) for season in by_season.index
]
by_season.round(4)

,model,market,gap,trained_seasons
Season,,,,
15-16,0.9902,0.9718,0.0184,3
16-17,0.9521,0.9338,0.0183,4
17-18,0.9582,0.9374,0.0208,5
18-19,0.9760,0.9487,0.0272,6
21-22,0.9976,0.9652,0.0324,7
22-23,0.9687,0.9580,0.0107,8
23-24,0.9632,0.9423,0.0209,9


In [11]:
descriptions = {
    "production-history": "today's model trained on 12-13 onwards; widened tune set (D9-1 baseline)",
    "market": "de-margined Pinnacle opening, Avg fallback (qH/qD/qA), history frame",
    "climatology": "each league's H/D/A base rates over the training seasons, history",
}
for name, scores in wide_scores.items():
    lab.log_variant(scores, TICKET, name, description=descriptions[name],
                    params={"history_start": "10-11", "train_start": lab.HISTORY_TRAIN_START})
lab.load_variants().query("ticket == @TICKET")[
    ["name", "period", "n", "log_loss", "market_log_loss", "gap", "gap_ci_low", "gap_ci_high",
     "draw_slope", "residual_weight"]
]

,name,period,n,log_loss,market_log_loss,gap,gap_ci_low,gap_ci_high,draw_slope,residual_weight
3,production-history,tune,4802,0.966024,0.950275,0.015749,0.010876,0.020838,3.271190,0.086103
4,production-history,tune_wide,16991,0.972334,0.951083,0.021250,0.018481,0.024029,2.750258,0.086103
5,market,tune_wide,16991,0.951083,0.951083,0.000000,0.000000,0.000000,1.136580,0.000000
6,climatology,tune_wide,16991,1.066426,0.951083,0.115342,0.109047,0.121787,0.219479,-0.277578


## Conclusions

**1. Like-for-like (22-23/23-24, 4,802 matches): the longer history is better.** Today's model, unchanged, trained on
12-13 onwards: log loss 0.9660 vs 0.9697, **-0.0037 [-0.0065, -0.0008]** (CI upper < 0), better in both seasons.
Most of it is in 22-23 (-0.0065), which today is predicted from one training season and starts from a cold Elo;
23-24 gains -0.0008. By league it helps D1, E0, I1, N1, P1 and SP1, and costs F1 a little (+0.0031).
The gap to the market narrows from **+0.0194 to +0.0157**. That's still far from the +0.010 "Good" level, and the
information tests still give the model nothing beyond the price (blend share 0.06; residual weight 0.086, CI
across 0; gain ~0).

**2. Widened tune set (15-16..18-19 + 21-22..23-24, 16,991 matches): the Phase 9 baseline.**
Model 0.9723 vs market 0.9511: **gap +0.0213 [+0.0185, +0.0240]**, CI half-width ~0.0028 (vs ~0.0052 on the two
tune seasons), so a 0.002-0.003 improvement can now be seen. Climatology is +0.115. The walk-forward takes ~20 s
(acceptance < 5 min).

**3. More data alone doesn't close the gap.** The per-season gap doesn't shrink as training seasons accumulate
(15-16 with 3 seasons: +0.018; 23-24 with 9: +0.021; 21-22, straight after COVID: +0.032). The remaining gap is
about features and model form (P9-02, P9-04..P9-08), not sample size.

**4. Draws are still the clearest fault.** The draw calibration slope is 2.75 on the wide set (3.27 on 22-23/23-24
with history vs 1.95 without): more data makes the sigmoid-calibrated LR's draw probabilities *flatter*.
This is P9-02's first target.

Caveats: xG before 20-21 is estimated (stats R^2 0.86, shots R^2 0.56), and P1 < 17-18 / N1 < 16-17 have none
(their `XGDDiffLast5` is 0 in training). The blend/residual tests are still fitted on 22-23/23-24 only.
P1 has the largest gap on the wide set (+0.032), partly because it has the least real xG.

Logged in `data/processed/model_variants.csv` (ticket P9-03): `production-history` on `tune` and `tune_wide`,
plus `market` and `climatology` on `tune_wide`. Later P9 tickets should score on `tune_wide` with
`matches=lab.history_matches()` and `frame=lab.history_frame()`, and compare against `production-history`.